

# Using Prolog to query databases
## Advanced Databases - INFO9016: Assignment 4 (tutorial)
---
**Alexandre Dutailly**
Email: alexandre.dutailly@student.uliege.be  
ID: s195855

**Thibaut Lefèvre**
Email: thibaut.lefevre@student.uliege.be  
ID: s164094

---


## Introduction

The SQL language, based on the relational model, remains the foundation of the majority of database management systems in production. However, when it comes to expressing complex recursive queries, manipulating fact graphs or integrating logical rules using non-monotonic negation, SQL quickly shows its limitations. It is precisely for these use cases that Prolog, and more specifically its SWI-Prolog implementation, comes into its own: knowledge is modelled in the form of facts and rules, and the engine relies on unification and backtracking to automatically explore the space of solutions, without the need for any loops or imperative structures.  

In addition to these basic mechanisms, Prolog offers fine-tuned ways of controlling the resolution process. The cut operator (`!`) is used to prune backtracking once a certain choice has been validated; a distinction is made between green cut, which improves performance without altering logic, and red cut, which modifies semantics by eliminating potential solutions. Failure negation, known as `\+`, is used to express that a fact cannot be proved, offering a non-classical form of negation that is useful in many reasoning scenarios.

To deal with combinatorial or planning problems, Prolog can call on the CLP(FD) library, which provides a finite-domain constraint solver and makes it very easy to formulate scheduling or allocation problems. Its system of modules also ensures that the code is clearly organised, and its interoperability with Python, via the PySWIP library, makes it easy to integrate Prolog into hybrid data pipelines where declarative processing and imperative scripts are combined.

In addition, Datalog, a subset of Prolog designed specifically for declarative interrogation of relational data, features a restricted syntax (with no cuts or compound terms) and a bottom-up fix-point evaluation that guarantees termination as soon as the data schema is finished. A closer look at Datalog will help you to place Prolog in the context of other logic languages and to understand why certain decision properties are sometimes preferable depending on the context.

In this tutorial, we will start by presenting the conceptual foundations of Prolog (facts, rules, unification, backtracking), illustrate the cut operators and failure negation, and then implement constraints with CLP(FD) on a concrete dataset. We will then introduce Datalog, comparing its evaluation mode and limitations with Prolog, before moving on to a practical thread in which the reader will use PySWIP to query a Pokémon database and explore these various mechanisms step by step.  

## References introduction :
- https://www.swi-prolog.org/pldoc/man?section=jitindex
- https://www.swi-prolog.org/pldoc/man?section=db
- https://github.com/yuce/pyswip

## Sources used for the introduction

| # | Référence | Lien |
|---|-----------|------|
| 1 | SWI-Prolog – *Control Predicates: !/0 (Cut)* :contentReference[oaicite:0]{index=0} | https://www.swi-prolog.org/pldoc/man?predicate=%21%2F0 |
| 2 | Learn Prolog Now! – *Negation as failure* :contentReference[oaicite:1]{index=1} | https://en.wikipedia.org/wiki/Negation_as_failure |
| 3 | SWI-Prolog – *library(clpfd): Constraint Logic Programming over Finite Domains* :contentReference[oaicite:2]{index=2} | https://www.swi-prolog.org/pldoc/man?section=clpfd |
| 4 | SWI-Prolog – *Modules system* :contentReference[oaicite:3]{index=3} | https://www.swi-prolog.org/pldoc/man?section=modules |
| 5 | PySWIP – *GitHub README* :contentReference[oaicite:4]{index=4} | https://github.com/yuce/pyswip/blob/master/README.md |
| 6 | Lecture 8 – *Introduction to Datalog* (UWisc CS784) :contentReference[oaicite:5]{index=5} | https://pages.cs.wisc.edu/~paris/cs784-f19/lectures/lecture8.pdf |
| 7 | Stack Overflow – *Differences between red cut and green cut* :contentReference[oaicite:6]{index=6} | https://stackoverflow.com/questions/25962835/prolog-differences-between-red-cut-and-green-cut |
| 8 | Wikipedia – *Datalog* :contentReference[oaicite:7]{index=7} | https://en.wikipedia.org/wiki/Datalog |


# 1. Installation

Here's a unified guide to preparing your environment, recommending that Windows users go through WSL2 (Ubuntu), then presenting the same procedure for macOS and for a native Ubuntu installation.

## 1.1. Windows : using Ubuntu with WSL2 ( or from google colabs )

### 1.1.1 Activate WSL2
```bash
wsl --install -d Ubuntu
wsl --set-version Ubuntu 2
```

### 1.1.2 Install SWI‑Prolog and Python in WSL2
In the Ubuntu window : 
```bash
sudo apt update
sudo apt install -y swi-prolog python3-venv python3-pip
```

### 1.1.3 Create environment from VSCode.
Open the WSL terminal from VSCode and run
```bash
code .
```
### 1.1.4 Install Jupyter and the Prolog bindings in the environment.
```bash
pip install --upgrade pip
pip install jupyterlab pyswip swiplserver
```
Checking the installation: 
```bash
swipl --version
python3 -c "from pyswip import Prolog; p=Prolog(); p.assertz('parent(a,b)'); print(list(p.query('parent(a,X)')))"
```

## 1.2. macOS : native installation
### 1.2.1 Install Homebrew (if needed)
```bash
/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"
```
### 1.2.2 Install SWI‑Prolog and Python.
```bash
brew update
brew install swi-prolog python
```
### 1.2.3 Create and activate a virtual environment Python
```bash
python3 -m venv venv
source venv/bin/activate
```
### 1.2.4 Install Jupyter and the Prolog bindings in the environment.
```bash
pip install --upgrade pip
pip install jupyterlab pyswip swiplserver
```
Check the installation : 
```bash
swipl --version
python3 -c "from pyswip import Prolog; p=Prolog(); p.assertz('parent(a,b)'); print(list(p.query('parent(a,X)')))"
```
## 1.3. Ubuntu
You need to create an environment and then follow the Windows tutorial from: `Installing Jupyter and Prolog bindings in the environment`.

# 2. Creating Prolog file
This part can be skipped in case you already have the pokemon.pl file downloaded, else you can run the following cells but it might take some time to run and write the file properly.  
The pokemon.pl file created in this section contains most of the facts we will use for this tutorial. The rules are implemented later in another file that is created during the tutorial.

## 2.1 Downloading the Pokémon List

We start by querying the PokeAPI endpoint `/api/v2/pokemon/` to retrieve the full list of available Pokémon. Each JSON response contains :

- **`count`**: total number of items in the collection.  
- **`results`** : an array of objects `{ ‘name’: ..., ‘url’: ... }` for each Pokémon on the page.

We use this API in **pagination** mode, in blocks of 100 entries, in order to collect all the names and associated URLs.

In [ ]:
# libraries used to download the data

import requests
import json
import time
from requests.exceptions import HTTPError
from pathlib import Path

In [ ]:
BASE_URL = "https://pokeapi.co/api/v2/pokemon/"

resp = requests.get(BASE_URL, params={"limit": 1})
resp.raise_for_status()
total = resp.json()["count"]
print(f"Count total : {total}")

all_pokemon = []
page_size = 100
for offset in range(0, total, page_size):
    r = requests.get(BASE_URL, params={"limit": page_size, "offset": offset})
    r.raise_for_status()
    for entry in r.json()["results"]:
        # extract ID from URL
        url = entry["url"].rstrip("/")
        pid = int(url.split("/")[-1])
        # only canonic species (pid < 10000)
        if pid < 10000:
            all_pokemon.append(entry)

print(f"Canonic species : {len(all_pokemon)}")

# saving file
with open("all_pokemon.json", "w", encoding="utf-8") as f:
    json.dump(all_pokemon, f, ensure_ascii=False, indent=2)

## 2.2 Retrieving Statistics and Types


From the `all_pokemon.json` list, we query the `/api/v2/pokemon/{name}/` detailed endpoint for each Pokémon, in order to retrieve :

- **Basic stats** (`hp`, `attack`, `defense`, ...).  
- **Types** (one or two categories per Pokémon).

The result is aggregated in a `detailed_data[name]` dictionary containing the `‘stats’` and `‘types’` keys.

In [ ]:
BASE_URL    = "https://pokeapi.co/api/v2/pokemon/"
SPECIES_URL = "https://pokeapi.co/api/v2/pokemon-species/"

def fetch_json(url, max_retries=3, backoff=0.5):
    """Récupère un JSON ou lève HTTPError après quelques tentatives."""
    for i in range(1, max_retries+1):
        r = requests.get(url)
        if r.status_code == 429:
            time.sleep(backoff * i)
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

# loading json file with the info
with open("all_pokemon.json", "r", encoding="utf-8") as f:
    all_pokemon = json.load(f)

detailed_data = {}
evolutions    = {}

for entry in all_pokemon:
    name = entry["name"]
    # Récupérer stats et types
    detail = fetch_json(f"{BASE_URL}{name}/")
    stats  = { s["stat"]["name"]: s["base_stat"] for s in detail.get("stats", []) }
    types  = [ t["type"]["name"] for t in detail.get("types", []) ]
    detailed_data[name] = {"stats": stats, "types": types}

    # Récupérer l'espèce de base pour l'évolution
    base_name = name.split("-", 1)[0]  # ex: "deoxys-normal" → "deoxys"
    try:
        species = fetch_json(f"{SPECIES_URL}{base_name}/")
        parent  = species.get("evolves_from_species")
        if parent and isinstance(parent, dict):
            evolutions[name] = parent["name"]
    except HTTPError:
        # Pas d'espèce trouvée (forme spéciale, nouveau format, etc.) : on ignore
        continue

print(f"Details extracted for {len(detailed_data)} Pokemon")
print(f"Number of evolution relationship : {len(evolutions)}")

# Sauvegarde
with open("detailed_pokemon.json", "w", encoding="utf-8") as f:
    json.dump(detailed_data, f, ensure_ascii=False, indent=2)
with open("evolutions.json", "w", encoding="utf-8") as f:
    json.dump(evolutions, f, ensure_ascii=False, indent=2)


## 2.3 Exporting Prolog facts

Finally, we convert `all_pokemon.json` and `detailed_pokemon.json` into a single `pokemon.pl` file, containing :
- `pokemon(Id, ‘Name’).`  
- `type(‘Name’, Type).`  
- `stat(‘Name’, Stat, Value).`  
where `Id` is taken from the original URL.

In [ ]:
all_pokemon = json.load(open("all_pokemon.json", "r", encoding="utf-8"))
detailed     = json.load(open("detailed_pokemon.json", "r", encoding="utf-8"))
evolutions   = json.load(open("evolutions.json", "r", encoding="utf-8"))

name2id = {
    entry["name"]: int(entry["url"].rstrip("/").split("/")[-1])
    for entry in all_pokemon
}
with open("pokemon.pl", "w", encoding="utf-8") as f:
    # directives discontiguous…
    f.write(":- discontiguous pokemon/2.\n")
    f.write(":- discontiguous type/2.\n")
    f.write(":- discontiguous stat/3.\n")
    f.write(":- discontiguous evolves_from/2.\n\n")

    # pokemon/2 : inchangé
    for entry in all_pokemon:
        name = entry["name"].replace("'", "\\'")
        pid  = name2id[name]
        f.write(f"pokemon({pid}, '{name}').\n")

    # type/2 : passer au pid
    f.write("\n")
    for name, info in detailed.items():
        pid = name2id[name]
        for t in info["types"]:
            f.write(f"type({pid}, {t}).\n")

    # stat/3 : idem, on écrit stat(pid, stat_name, val)
    f.write("\n")
    for name, info in detailed.items():
        pid = name2id[name]
        for stat_name, val in info["stats"].items():
            f.write(f"stat({pid}, '{stat_name}', {val}).\n")

    # evolves_from/2 : on transforme aussi parents/enfants en pid
    f.write("\n")
    for child, parent in evolutions.items():
        f.write(f"evolves_from('{child}', '{parent}').\n")

print("pokemon.pl generated with succes")


https://publicapi.dev/pokeapi-api

## 3. Prolog is a declarative logic programming language
Prolog is a declarative logic programming language that follows first order logic. It is similar to _Datalog_ that has been seen during the _advanced databases_ course. However, as we will see further, it has differences with _Datalog_, being a generalization of the latter, it offers more possibilities, coming with additional constraints.  
In this section, the different mechanisms of prolog will be explained from a theoritical point of view. These mechanisms will be illustrated later in section 4 using Pokemon examples written with **SWIProlog**.  
Although the basics are covered, it is assumed that the knowledge on deductive databases from chapter one and exercise session from the _advanced databases_ course have been acquired.  
This part is heavily based on the online version of _Learn Prolog now!_ [freely accessible here](https://lpn.swi-prolog.org/lpnpage.php?pageid=online), and which is recommended if the user wants a deeper theoritical understanding of the covered topics or extra exercises.  
  
In this tutorial, we use SWI-Prolog implementation of Prolog and the python interface PySwip.

In [ ]:
# libraries and instantiation

from pyswip import Prolog

prolog = Prolog()

# Loading the knowledge base created earlier
prolog.consult('pokemon.pl')

# Querying the list of Pokemon in our knowledge base
pokemons = prolog.query('pokemon(_,X)')
pokemon_in_kb = [poke['X'] for poke in pokemons]
pokemon_in_kb

If you want to directly use the terminal directly instead of the python interface, you can start a session by running the command `$ swipl`. It is advised to run the terminal in the same directory as your documents are.  
If you want to end the session, use the command `halt.` (do not forget the dot `.`)  
  
To load our file `pokemon.pl`, run the command `[pokemon]` (your terminal needs to run in the same directory as your files, else you need to specify the path)

### 3.1 Syntax
Similarly to _Datalog_, knowledge bases in _Prolog_ are composed of _facts_ and _rules_ and consititute the Prolog program in itself. We will then query the base using _queries_.  
Here are some examples of _facts_, _rules_ and _queries_ taken from the Pokemon example:  
- `pokemon(4, charmander).` is a _fact_, which states that charmander is a pokemon whose id is 4 (the predicate pokemon, on the number 4 and the constant atom charmander is true) 
- `ancestor(Poke, Anc) :- evolves_from(Poke, Anc).` is a _rule_ that states that the predicate ancestor is true between two variables if the predicate evolves_from is true between those two variables.
- `? - pokemon(Id, Name), Id =< 151.`is a _query_. This _query_ is asking the Pokemon names and id's for which the id is below or equal to 151 (corresponding to first gen Pokemon's ;) )  
Notice that both _fact_, _rules_ and _queries_ end with a point `.` and that predicates and constant atoms start with a lowercase letter whereas variables start with an uppercase letter.  

In [ ]:
list(prolog.query('pokemon(4, charmander)')) # It has to be noted that with pyswip we will not have a yes - no answer as we would if we used the terminal

#### Type of terms:
There are 3 types of terms in prolog: 
- Constant terms (either atoms or numbers)
- Variables
- Complex terms, which here differentiate _Prolog_ from _Datalog_ as the complex term from prolog have more complex structures that _Datalog_ can't handle, such as lists `[grovyle, zangoose, rayquaza, kyogre, groudon, salamence]` or imbricated terms 

#### Conjunction and disjunction:
- Conjunction (AND) between two different goals is denoted `,`, for example: `type(253, grass).`. 
- Disjunction (OR) between two different goals is denoted `;`, for example: `? - (stat(Id, attack, Atk), Atk >= 90) ; (stat(Id, special-attack, SpAtk), SpAtk >= 90).`.

### 3.2 Unification and proof search
#### Unification
A fundamental difference between _Datalog_ and _Prolog_ is the way of searching a proof. This will cause major difference between the two languages such as the order of the clauses having an impact on the results in _Prolog_, which will require attention when writing rules. To understand this, let's start with the notion of unification: Pokemon = grovyle.  
Two terms will unify either if they are the same (in the case of constant terms) or if the variables in the terms can be instanciated such that the resulting terms are equal.  
As an example, the query `? - grovyle = grovyle.` will return `yes`, while `? grovyle = mudkip.` will return `no`. When variables are involved such as `? - grovyle = Pokemon.`, _Prolog_ will try to unify the variable Pokemon to a constant that validates this query, it will return `yes`and `grovyle = Pokemon`.  
On an unrelated note (as it is not developped in depth in this tutorial), arithmetical expressions do not work with the `=` unification operator: `? - 4 = 2+2.` will return no, to take into account the value behind 2+2, you must the arithmetic equality operator `=:=`, such that `? - 4 =:= 2+2.` returns yes


#### Proof search and backtracking
When queried, _Prolog_ will look for a solution using a _Depth first search_ algorithm and backtrack when it cannot go further. The search is top to down, left ro right. Which means that the order of the rules have an incidence on the solution.  
  
For our example, let's imagine that we want to find the name of the water type evolution of Eevee:  
&emsp;`? - evolves_from(Watereeveelution, eevee), pokemon(Id, Watereeveelution), type(Id, water).`  
  
_Prolog_ will attempt to satisfy the different goals by exploring different values the variable Watereeveelution can have. The approach is _Depth first search_ because it tries a binding and will attempt to satsify all goals before exploring the next binding. It is left to right because the goals are tested from left to right.  
In our example, let's imagine that the first explored binding for Watereeveelution is gyarados. The first goal is not satisfied as evolves_from(gyarados, eevee) is not a fact from our program. _Prolog_ will jump to another binding. Let's imagine now that the next binding is jolteon, the first goal will be satisfied, as evolves_from(joltean, eevee) is true, it will therefore jump to the next goal, which will instanciate an Id value and then jump to the third goal which returns no. Since a goal has not been satisfied, another binding is explored and it will do so up until all possible bindings are explored or a solution is found.   
  
This method of proof search makes the program dependent on the order of the clauses, which is not the case in _Datalog_ and makes an important difference between the two. Generally, the order of the facts only influences the results of some clauses. On the other hand, the order of the rules can allow/prevent the program to run properly, creating infinite loops !  

Here is a schema illustrating our example about backtracking
  
```text
?- evolves_from(Watereeveelution, eevee),
   pokemon(Id, Watereeveelution),
   type(Id, water).

evolves_from(Watereeveelution, eevee)
├── Watereeveelution = gyarados
│   └── evolves_from(gyarados, eevee) ✗ (fails immediately)
│
├── Watereeveelution = jolteon
│   ├── evolves_from(jolteon, eevee) ✓
│   ├── pokemon(Id = 135, jolteon)   ✓
│   ├── type(135, water)             ✗
│   └── backtrack
│
├── Watereeveelution = vaporeon
│   ├── evolves_from(vaporeon, eevee) ✓
│   ├── pokemon(Id = 134, vaporeon)   ✓
│   ├── type(134, water)              ✓
│   └── ✔ success → Watereeveelution = vaporeon, Id = 134
│
└── (other possibilities explored if more answers are requested)
```

_Schema generated using ChatGPT_

### 3.3 Recursion
_Prolog_, as _Datalog_ does, handles recursion. However, due to their differences in the way they do proof search, the implementation of recursion in _Prolog_ requires extra constraints.  
Let's look at a pokemon example:
    
In this example we are looking at Pokemon's evolutions. In case you are not familiar with pokemon, you can interpret this example as some parent - ancestor relationship, pokemons are evolving (transforming) into another one, and might later evolve again, we want ancestor/2 to determine if a pokemon is an evolution of another or one of his future evolutions.  
  
&emsp;`ancestor(Poke, Anc) :- evolves_from(Poke, Anc).`  
&emsp;`ancestor(Poke, Anc) :- evolves_from(Poke, Mid), ancestor(Mid, Anc).`  
  
As you may notice, we have here two clauses: the _base clause_, which is written first and explains the base case (`ancestor/2` only appears in the head of the rule) and the _recursive clause_, which contains the recursion (`ancestor/2`, appears in the head, but also in the body of the rule). Again, the order of the clauses is important and the _recursive clause_ being written first would, make the program slower, or create infinite loops, it is therefore required to write the _base clause_ first.  
In our example, _Prolog_ would try to bind all possibles values to the second goal of the _recursive clause_ without ever being able to validate it as the _base clause_ is written after the recursive one.

### 3.4 Lists
Lists are a key feature of _Prolog_ and an important difference with _Datalog_ as they do not exist in that language. A list is a sequence of elements, similar to what you might encounter in other programming languages, they support nesting (element of a list being a list as well) and are written using suqare brackets as following:  
`[treecko, torchic, [geodude, nosepass], ralts].`  
This list for example contains the pokemons of Thibaut, Alexandre, Roxanne, the gym leader of Rustboro City's gym and Wally. You can see that the third elements of the list (Roxanne's Pokemons) is a list as well and occupies a single position. Therefore, the pokemon of Wally, ralts, which comes after the list of pokemon of Roxanne is the fourth element of the list. The presence of a nested list does not affect the indexing of the list.  

#### Head and tail in a list: list constructor operator |
A specificity of lists is _Prolog_ is the presence of the list cons operator `|`which separates a list between its _head_ and its _tail_. 
- the _head_ of a list is its very first element.
- the _tail_ of a list is everything that is after the _head_  
  
Using our previous pokemon example:    
&emsp;`? - [treecko, torchic, [geodude, nosepass], ralts].  = [H|T].`  
&emsp;_Prolog_ will unify `H` (the head) with `treecko` while `T` (the tail) will be unified with `[torchic, [geodude, nosepass], ralts]`.  
  
 . 
    
You are also not limited to getting only the very first element of the list. Using the list cons operator properly, you can extract more than the very first element by putting more than one variable before the list cons operator. Let's see it in our example where we want to obtain the first two elements of the list:  
&emsp;`? - [treecko, torchic, [geodude, nosepass], ralts].  = [H1, H2|T].`   
&emsp;`H1` is unified with `treecko`, `H2` with `torchic` and `T` with `[[geodude, nosepass], ralts]`, the first two elements have therefore been exstracted !
  
It is also noteworthy to take into account that the list is required to contain elements to have a head an a tail:  
&emsp;`? - [H|T] = []` returns no as the list is empty  
&emsp;`? - [treecko, torchic, [geodude, nosepass], ralts].  = [H1, H2, H3, H4|T].` In this case, `T` is unified with [] whereas the H variables are unified with the elements of the list.  
&emsp;`? - [treecko, torchic, [geodude, nosepass], ralts].  = [H1, H2, H3, H4, H5|T].` Returns no, as H5|T is similar to trying to get the _head_ and _tail_ of an empty list.

#### Applying rules on lists
A crucial point that has not been tackled yet is the incorporation of lists into rules. To deal with it, _Prolog_ will use recursion to examine each element of the list. Let's have an example coming from [the chapter 4.2 of _Learn Prolog Now!_](https://lpn.swi-prolog.org/lpnpage.php?pagetype=html&pageid=lpn-htmlse14)  
  
Let us imagine that we have a list consisting of the pokemons of Alexandre (who, as you can see, has improved his team since the last example):  
&emsp;`[combusken, wobbuffet, feebas, slakoth, zigzagoon].`  
  
We want to know whether or not his team contains the pokemon `rayquaza`, to verify it, the following recursive rules will be written:  
&emsp;`member(Pokemon, [Pokemon|T]).` as the base clause: if a pokemon is the tail of the list, it belongs to the team. Let's now introduce the recursive rule:  
&emsp;`member(Pokemon, [H|T]) :- member(Pokemon, T).` which contains recursion and will "parse" the list to see if the Pokemon belongs to it.  
  
A more complex example based on Pokemon battles using lists will be detailed in section 4.

### 3.5 Cut operator  `!`
The cut operator is used to control the backtracking. It is used to improve the efficiency of the program (_green cuts_), but can also change the results (_red cuts_), which is not recommended.  
It is used in the body of a clause, when called (meaning that the previous goal has succeeded), it cuts all further backtracking (further clauses and or binding for variables are not explored) and commits to the choices already made.  

Using another Pokemon example, let's imagine that we want to create a rule to determine the generation of a pokemon (to not make the rule too long, we will only consider the case where the pokemon belongs to one of the 3 first generations or a following generation) 
  
```prolog
pokemongen(Pokemon, gen1) :- 
    pokemon(Id, Pokemon),
    Id =< 151.

pokemongen(Pokemon, gen2) :- 
    pokemon(Id, Pokemon),
    Id > 151,
    Id =< 251.

pokemongen(Pokemon, gen3) :- 
    pokemon(Id, Pokemon),
    Id > 251,
    Id =< 386.

pokemongen(Pokemon, past3rdgen) :- 
    pokemon(Id, Pokemon),
    Id > 386.
```

It can also be written the following way if we want all the logic in a single rule (we won't use thise writing for our example)
```prolog
pokemongen(Pokemon, Gen) :- pokemon(Id, Pokemon), (
    (Id =< 151, Gen = gen1); 
    (Id > 151, Id =< 251, Gen = gen2); 
    (Id > 251, Id =< 386, Gen = gen3); 
    (Id > 386, Gen = past3rdgen)
    ).
```

#### _Green cut_
In this example, if we want to know the generation Lugia belongs to with the query `? - pokemongen(lugia, Gen).`, _Prolog_ will try each rule in turn to determine all the possible generations for Lugia. However, we know that a pokemon can belong to only one generation. As soon as a rule succeeds, we know that the other ones fail. There is in that case no need to explore them. The cut operator allows us to stop the unnecessary exploration of rules after a successful match.  
  
Let's incorporate the cut operator `!` in our rules to make the program more efficient:  
```prolog
pokemongen(Pokemon, gen1) :- 
    pokemon(Id, Pokemon),
    Id =< 151, !.

pokemongen(Pokemon, gen2) :- 
    pokemon(Id, Pokemon),
    Id > 151,
    Id =< 251, !.

pokemongen(Pokemon, gen3) :- 
    pokemon(Id, Pokemon),
    Id > 251,
    Id =< 386, !.

pokemongen(Pokemon, past3rdgen) :- 
    pokemon(Id, Pokemon),
    Id > 386, !.
```
Using these rules, the answer `? - pokemongen(lugia, Gen).` does not change if the order in which the rules are written is different. The incorporation of cuts only impacts the efficiency of the program. This is an example of a _green cut_.

Here is another schema illustrating our example:
    
```text
?- pokemongen(lugia, Gen).

pokemongen(lugia, Gen)
├── Rule 1: gen1
│   ├── pokemon(249, lugia) ✓
│   ├── 249 =< 151         ✗
│   └── fails, try next rule
│
├── Rule 2: gen2
│   ├── pokemon(249, lugia) ✓
│   ├── 249 > 151           ✓
│   ├── 249 =< 251          ✓
│   ├── ! ← cut activated here
│   └── ✔ success → Gen = gen2
X    ↳ Prolog stops here — no further rules are tried
```
  
_Schema generated using ChatGPT_

#### _Red cut_
On the other hand, let's imagine that we directly created the rules with the intent to use the cut operator and have our rules written the following way:
```prolog
pokemongen(Pokemon, gen1) :- 
    pokemon(Id, Pokemon),
    Id =< 151, !.

pokemongen(Pokemon, gen2) :- 
    pokemon(Id, Pokemon),
    Id =< 251, !.

pokemongen(Pokemon, gen3) :- 
    pokemon(Id, Pokemon),
    Id =< 386, !.

pokemongen(Pokemon, past3rdgen) :- 
    pokemon(Id, Pokemon), !.
```

Here, the clause for pokemon(Pokemon, gen2) does not have the goal Id > 151 because it is intended that the Id is already over 151 if we reach that point, else it would have satisfied the clause pokemon(Pokemon, gen1). However, if we change the order of the clauses in the program, the solutions will be affected: if pokemongen(Pokemon, past3rdgen) is the first clause, `past3rdgen` will always be a solution for an existing pokemon using `? - pokemongen(Pokemon, Gen).`.  
This is an example of a red cut as the solution depends on the order of the rules. This practice is not recommended.

#### Writing the pokemongen rules with green cuts in our file:
Although it is possible to write our rules in our file by editing it manually, we will do so through python for the tutorial such that you do not have to leave the notebook

In [ ]:
# Rules that we want to write
rules_gen = """
pokemongen(Pokemon, gen1) :- 
    pokemon(Id, Pokemon),
    Id =< 151, !.

pokemongen(Pokemon, gen2) :- 
    pokemon(Id, Pokemon),
    Id > 151,
    Id =< 251, !.

pokemongen(Pokemon, gen3) :- 
    pokemon(Id, Pokemon),
    Id > 251,
    Id =< 386, !.

pokemongen(Pokemon, past3rdgen) :- 
    pokemon(Id, Pokemon),
    Id > 386, !.
"""

# Writing the rules
with open("pokemon_rules.pl", "a", encoding="utf-8") as f:
    f.write("\n" + rules_gen.strip() + "\n")

# Consulting the file
prolog.consult("pokemon_rules.pl")

### 3.6 Negation as Failure

Negation as Failure (NaF), also written as `\+ P` or `not(P)`, is the mechanism that Prolog uses to introduce a form of negation in the absence of a conventional ‘¬’ operator. It is based on the idea that any fact that cannot be demonstrated from the knowledge base is considered false: this is the **closed world hypothesis**. In concrete terms, when Prolog encounters ``P‘’, it first tries to prove the objective ``P‘’. If this attempt fails, `P' is validated; if it succeeds, the negation fails.  
  
Negation as Failure is implemented using the cut operator seen earlier. The goal `\+ type(Id, water)` (in case we are looking for a pokemon that is not of water type) could be rewritten in a clause using cut operator `type(Id, water), !, fail` such that if at any moment `type(Id, water)` succeeds, the clause fails and the backtracking is stopped.

This approach is intrinsically **non-monotonic**: the addition of a new fact or rule to the base can transform a valid negation into an invalid one, since proof of the negative objective may become possible. From a theoretical point of view, failure negation is **sound** (it doesn't generate false positives) but **incomplete** (it doesn't always discover all the true facts), and its behaviour can become tricky if we try to negate objectives containing uninstantiated variables. In this case, Prolog stops as soon as it finds the first partial solution for `P`, which can lead to flounderings or unexpected results.  
  
In summary, Prolog's failure negation functions as a proof failure test, based on the closed world hypothesis and characterised by its non-monotonicity. It provides a simple and powerful way of expressing negation in a reduced first-order logic language, while remaining limited by its incompleteness and the handling of non-ground variables.  

### 3.7 Constraint programming: `library(clpfd)


The CLP(FD) (Constraint Logic Programming over Finite Domains) module extends Prolog by giving it access to a constraint solver over finite sets of integers. Rather than dealing with arithmetic and logical relations only at the time of unification or backtracking, CLP(FD) allows a discrete domain of admissible values to be declared for each variable from the outset, using the `Var ins Low...High` instruction. When a constraint is imposed, such as `X #< Y` or `X #\= 5`, the solver acts immediately: it reduces the bounds of the domains of all the variables concerned, by arc-consistency propagation, until it reaches a fixed point where no new restrictions can be applied.  


If propagation alone is not enough to assign all the variables, we then move on to a ‘labelling’ phase, which chooses a variable that has not yet been instantiated, according to a built-in heuristic (by default ‘first-fail’, i.e. the most restricted), and successively assigns it each possible value in its domain. At each attempt, propagation is automatically restarted, which quickly eliminates impossible choices. In the event of failure, Prolog returns to the last decision point and tries the next value, while retaining the domain refinements already acquired, which makes the search much more efficient than a simple exhaustive traversal.  
  
In short, `library(clpfd)` transposes the main principles of constraint programming into Prolog: the problem is described in terms of domains and declarative relations, and the engine automatically propagates the restrictions and manages the search with intelligent backtracking. This approach is particularly powerful for planning and scheduling problems, combinatorial puzzles (such as Sudoku or generalised Towers of Hanoi) and any task requiring optimised exploration of a space of finite solutions.   

### 3.8 Modules and encapsulation
In Prolog, a **module** is a unit of code which groups together a set of predicates under a single namespace and explicitly defines a public interface. A module is declared with :

```prolog
:- module(module_name, [pred1/arity1, pred2/arity2, ...]).
```

The predicates listed are exported, i.e. accessible from the outside; all the others remain private, ensuring true encapsulation of implementation details. To use the functionality of another module, import it with :

```prolog
:- use_module(library(lists)).   % imports all predicates exported from library(lists)
```

or more precisely :

```prolog
:- use_module(library(lists), [member/2, append/3]).   % only imports member/2 and append/3
```

This interface/implementation separation means that the internal implementation can be modified without impacting the client code, auxiliary predicates can be hidden and dependencies between files can be clearly managed. By isolating the names of predicates within a module, we can also avoid name collisions in large projects, making the code easier to read and maintain.

### 3.9 Tabling (memoization)


Tabling** in Prolog, also known as **SLG resolution**, activates a resolution mode which combines the classic **top-down** approach with controlled **bottom-up** processing. For a predicate to benefit from this mechanism, all you have to do is annotate it:


```prolog

:- table my_predicate/2.

```

At runtime, Prolog stores in a table each unique (subgoal) call to my_predicate/2 and the results it produced. If the same call reappears, Prolog does not recompute it: it reuses the stored responses directly, avoiding recomputations and guaranteeing termination for many forms of cyclic recursion or multi-path graphs. This strategy provides a superset of Datalog semantics and limits the non-termination problems of traditional Prolog .


The order in which you write your clauses and sub-objectives has no influence on the completeness of the solutions collected (all are collected regardless of their position). On the other hand, it does play a crucial role in performance and convergence: placing the basic clauses (facts or non-recursive rules) first and ordering the calls so as to produce elementary results quickly allows the engine to saturate the table sooner, thus reducing the number of temporary entries to be managed. Poor ordering can multiply the size of tables and slow down resolution.

## Sources for section 3

| # | Référence | URL |
|---|-----------|-----|
| 1 | Single sided unification & backtracking – SWI-Prolog :contentReference|https://www.swi-prolog.org/pldoc/man?section=ssu
| 2 | Backtracking necessity in Prolog proof search – StackOverflow :contentReference |https://stackoverflow.com/questions/25962835/prolog-differences-between-red-cut-and-green-cut
| 3 | Green vs Red cut – StackOverflow :contentReference |https://stackoverflow.com/questions/14079805/how-does-negation-as-failure-works-in-prolog
| 4 | Negation as failure & closed-world assumption – StackOverflow :contentReference |https://lpn.swi-prolog.org/lpnpage.php?pagetype=html&pageid=lpn-htmlse45
| 5 | CLP(FD) overview – SWI-Prolog :contentReference |https://www.swi-prolog.org/pldoc/man?section=clpfd
| 6 | Modules system – SWI-Prolog :contentReference |https://www.swi-prolog.org/pldoc/man?section=tabling
| 7 | Table/1 directive & SLG resolution – SWI-Prolog :contentReference |https://www.swi-prolog.org/pldoc/man?section=ssu
| 8 | Learn Prolog Now! |https://lpn.swi-prolog.org/lpnpage.php?pageid=online
| 9 | *Zhang, H. (n.d.). Cut Operator in Prolog - Logic in computer science* University of Iowa |https://homepage.divms.uiowa.edu/~hzhang/c188/notes/ch08b-cut.pdf

# 4. Step-by-step tutorial: querying the Pokémon database with **SWI-Prolog**
> The following cells assume that the **`pokemon.pl`** file (generated in section 3) is located in the same folder as the notebook.  
> All Prolog requests are executed via **PySWIP**; each Python block is therefore directly reproducible.

---
## 4.1 Loading facts and quick verification

In [ ]:
from pyswip import Prolog
prolog = Prolog()

prolog.consult("pokemon.pl")

nb = len(list(prolog.query("pokemon(_, _)")))
print(f"Loaded Pokemon : {nb}")

In [ ]:
sol = next(prolog.query(
    "setof(Type, Id^type(Id,Type), Types),"
    "setof(Stat, I^V^stat(I,Stat,V), Stats)"
))
print("number of types:", len(sol["Types"]))
print("Types available:", sol["Types"])
print("Numbers of stats available:", len(sol["Stats"]))
print("Statistics available:", sol["Stats"])


## 4.2 Basic queries


We start with some simple queries to familiarise the reader with Prolog syntax and the unification method:


- **Filter by generation**: extract Pokémon from the first generation (ID ≤ 151).  

- **Filter by statistic**: list all Pokémon whose basic life value (`hp`) exceeds a given threshold.


This step illustrates the power of declarative unification and backtracking to find subsets of the database.

### 4.2.1 Finding first gen pokemon
For this example, we do not the rule `pokemongen(Pokemon, gen1)` that has been mentionned earlier

In [ ]:
for pok in prolog.query("pokemon(Id, Name), Id =< 151", maxresult=10):
    print(pok)

### 4.2.Listing Pokémon whose hp are over 100

In [ ]:
solutions = list(prolog.query("stat(Name, hp, HP), HP > 100"))
print(f"Number of Pokémon with HP > 100 : {len(solutions)}")
print([sol["Name"] for sol in solutions[:10]])

## 4.3 Combined queries (logical joins)
We now move on to queries which combine several predicates:
- **Use case**: find the ‘tanks’ of a given type (for example `steel`) whose hit points exceed a certain threshold.  
- Definition of a `tank_type/3` predicate combining `type/2`, `stat/3` and `pokemon/2`.
This section demonstrates Prolog's ability to perform logical joins without a DISTINCT clause or nested subqueries.

In [ ]:
from pathlib import Path

rules = """
:- discontiguous fetch_stat/3.
:- discontiguous team_2stat/5.
:- discontiguous tank_type/4, ancestor/2,
               first_elec_nocut/1, first_elec_cut/1,
               not_water/2.
               
:- use_module(library(clpfd)).
:- use_module(library(statistics)).
:- table ancestor/2.


tank_type(Type, Id, Name, HP) :-
    type(Id, Type),
    stat(Id, hp, HP),
    HP > 120,
    pokemon(Id, Name).

   """
fpath = Path("pokemon_rules.pl")
with fpath.open("a") as f:
            f.write(rules)
prolog.consult("pokemon.pl") 
prolog.consult("pokemon_rules.pl")

In [ ]:
solutions = list(prolog.query("tank_type(steel, Id, Name, HP)", maxresult=10))
print(f"Number of steel Pokémon with over 120 hp : {len(solutions)}")
solutions

## 4.4 Recursive query: evolution chain
To demonstrate the expressiveness of recursive rules, we define the transitive relation :
- `ancestor/2` from the predicate `evolves_from/2`.  
- Illustration of recursive resolution, and demonstration of multiple backtracking in the case of long evolution chains.
This is a key example where Prolog excels compared to SQL, because recursion is a native use case.

In [ ]:
rules = """
ancestor(Poke, Anc) :- evolves_from(Poke, Anc).
ancestor(Poke, Anc) :- evolves_from(Poke, Mid), ancestor(Mid, Anc).
   """
fpath = Path("pokemon_rules.pl")
with fpath.open("a") as f:  # mode append
            f.write(rules)
prolog.consult("pokemon.pl")          # facts
prolog.consult("pokemon_rules.pl")    # rules and index

In [ ]:
for sol in prolog.query("ancestor(raichu, Anc)"):
    print("Raichu is the evolution of :", sol["Anc"])

## 4.5 Flow control: demonstrating the cut
We compare two versions of the same predicate:
- **Without cut**: full exploration of all possible solutions.  
- **With green cut**: backtracking from the first solution, without modifying the set of valid results.  
- **With red cut** (brief illustration): to show how a badly placed cut can eliminate solutions.
The aim is to measure the reduction in the number of inferences or search time, using statistical predicates.

The poke_role example was originally not a green cut during the presentation, if poke_role(Id, weak) is evaluated first it would return True for every pokemon because the other roles have not been checked yet. Therefore, some additional goals incorporating negation as failure were implemented such that, poke_role(Id, weak)  does not return True if the pokemon has another role than weak. It is still to be noted that this implementation is still not really efficient if poke_role(Id, weak) is evaluated first as every other clause would be called as well.

In [ ]:
rules = """
first_type_nocut(Type, Name) :-
    type(Id, Type),
    pokemon(Id, Name).

first_type_cut(Type, Name) :-
    type(Id, Type),
    pokemon(Id, Name),
    !.

poke_role(Id, sweeper) :-
    stat(Id, speed, S), S > 100,
    stat(Id, attack, A), A > 90,
    !.

poke_role(Id, tank) :-
    stat(Id, hp, HP), HP > 120,
    stat(Id, defense, D), D > 90,
    !.

poke_role(Id, balanced) :-
    stat(Id, attack, A), A >= 60, A =< 90,
    stat(Id, defense, D), D >= 60, D =< 90,
    !.

poke_role(Id, weak) :-
    \+ poke_role(Id, sweeper),
    \+ poke_role(Id, tank),
    \+ poke_role(Id, balanced), 
    !.


poke_role_named(Name, Role) :-
    pokemon(Id, Name),
    poke_role(Name, Role).
    """

fpath = Path("pokemon_rules.pl")
with fpath.open("a") as f:
            f.write(rules)
prolog.consult("pokemon.pl")    
prolog.consult("pokemon_rules.pl")    

In [ ]:
sols_nocut = list(prolog.query("first_type_nocut(water,Name)"))
sols_cut = list(prolog.query("first_type_cut(water,Name)"))
role1 = list(prolog.query("poke_role_named(Name, Role)", maxresult=10))
print(len(sols_nocut), sols_nocut)
print(len(sols_cut),   sols_cut)
for r in role1:
    print(f"{r['Name']} => {r['Role']}")

## 4.6 Negation as Failure (``+`)
Let's look at how Prolog handles non-monotonic negation:
- Define a `not_type/2` predicate that lists Pokémon **not** type `water`.  

In [ ]:
rules = """
not_type(Id, Name, Type) :-
    pokemon(Id, Name),
    \+ type(Id, Type).
    """
fpath = Path("pokemon_rules.pl")
with fpath.open("a") as f:
            f.write(rules)
prolog.consult("pokemon.pl")    
prolog.consult("pokemon_rules.pl")    

In [ ]:
list(prolog.query("not_type(Id, Name, water)", maxresult=10))

## 4.7 Constraint programming : CLP(FD)
We use the **CLP(FD)** module to solve a small **team composition** problem:
- Choose a fixed number of Pokémon whose sum of HP and/or attacks respects arithmetic constraints.  
- Set up the domains, propagate the constraints and find a declarative solution.  

This demonstration illustrates the power of Prolog for combinatorial problems, without recourse to an external language.

In [ ]:
rules = """
team_2stat(Stat1, Min1, Stat2, Max2, Ids) :-
    length(Ids, 6),
    Ids ins 1..1025,
    all_distinct(Ids),

    % Récupérer les valeurs pour chaque stat
    maplist(fetch_stat(Stat1), Ids, Vs1),
    maplist(fetch_stat(Stat2), Ids, Vs2),

    % Contraintes sur les sommes
    sum(Vs1, #>=, Min1),
    sum(Vs2, #=<, Max2),

    label(Ids).
fetch_stat(StatName, _Id, Value) :-
    stat(_Id, StatName, Value).
   """
fpath = Path("pokemon_rules.pl")
with fpath.open("a") as f:  # mode append
            f.write(rules)
prolog.consult("pokemon.pl")          # facts
prolog.consult("pokemon_rules.pl")    # rules index

In [ ]:
for sol in prolog.query("team_2stat(hp, 600, attack, 550, Ids).", maxresult=1):
    print("Équipe trouvée :", sol["Ids"])

## 4.8 Pokemon battles using lists
In this example, we try to emulate Pokemon battles between two trainers their teams using lists.  
  
First, let's write the rules in the file, they will next be explained step by step.  

In [ ]:
rules_battle = """
trainerwithteam(thib, [grovyle, zangoose, rayquaza, kyogre, groudon, salamence]).
trainerwithteam(alex, [combusken, wobbuffet, feebas, slakoth, zigzagoon]).

totstats(Id, Totstat) :-
    stat(Id, 'hp', Hp),
    stat(Id, 'attack', Atk),
    stat(Id, 'special-attack', SpAtk),
    stat(Id, 'special-defense', SpDef),
    stat(Id, 'defense', Def),
    stat(Id, 'speed', Spd),
    Totstat is Hp + Atk + SpAtk + Def + SpDef + Spd.

battleteam(Team1, Team2, team1wins, _, _) :- Team2 = [],  Team1 \= [].
battleteam(Team1, Team2, team2wins, _, _) :- Team1 = [],  Team2 \= [].
battleteam(Team1, Team2, draw, _, _) :- Team1 = [],  Team2 = [].

battleteam([HT1|TT1], [HT2|TT2], Winningteam, C1, _) :-
    pokemon(Id1, HT1),
    totstats(Id1, Stat1),
    pokemon(Id2, HT2),
    totstats(Id2, Stat2),
    Stat1 > Stat2,
    C1 < 2,
    C1Next is C1 + 1,
    battleteam([HT1|TT1], TT2, Winningteam, C1Next, 0).

battleteam([HT1|TT1], [HT2|TT2], Winningteam, C1, _) :-
    pokemon(Id1, HT1),
    totstats(Id1, Stat1),
    pokemon(Id2, HT2),
    totstats(Id2, Stat2),
    Stat1 > Stat2,
    C1 >= 2,
    battleteam(TT1, TT2, Winningteam, 0, 0).

battleteam([HT1|TT1], [HT2|TT2], Winningteam, _, C2) :-
    pokemon(Id1, HT1),
    totstats(Id1, Stat1),
    pokemon(Id2, HT2),
    totstats(Id2, Stat2),
    Stat1 < Stat2,
    C2 < 2,
    C2Next is C2 + 1,
    battleteam(TT1, [HT2|TT2], Winningteam, 0, C2Next).

battleteam([HT1|TT1], [HT2|TT2], Winningteam, _, C2) :-
    pokemon(Id1, HT1),
    totstats(Id1, Stat1),
    pokemon(Id2, HT2),
    totstats(Id2, Stat2),
    Stat1 < Stat2,
    C2 >= 2,
    battleteam(TT1, TT2, Winningteam, 0, 0).

battleteam([HT1|TT1], [HT2|TT2], Winningteam, _, _) :-
    pokemon(Id1, HT1),
    totstats(Id1, Stat1),
    pokemon(Id2, HT2),
    totstats(Id2, Stat2),
    Stat1 =:= Stat2,
    battleteam(TT1, TT2, Winningteam, 0, 0).

trainersbattle(Trainer1, Trainer2, Trainer1) :-
    trainerwithteam(Trainer1, Team1),
    trainerwithteam(Trainer2, Team2),
    battleteam(Team1, Team2, team1wins, 0, 0), !.

trainersbattle(Trainer1, Trainer2, Trainer2) :-
    trainerwithteam(Trainer1, Team1),
    trainerwithteam(Trainer2, Team2),
    battleteam(Team1, Team2, team2wins, 0, 0), !.

trainersbattle(Trainer1, Trainer2, draw) :-
    trainerwithteam(Trainer1, Team1),
    trainerwithteam(Trainer2, Team2),
    battleteam(Team1, Team2, draw, 0, 0), !.
"""

fpath = Path("pokemon_rules.pl")
with fpath.open("a") as f:  # mode append
            f.write(rules)
prolog.consult("pokemon.pl")          # facts
prolog.consult("pokemon_rules.pl")    # rules index

Let us start with the following facts:  
```prolog
trainerwithteam(thib, [grovyle, zangoose, rayquaza, kyogre, groudon, salamence]).
trainerwithteam(alex, [combusken, wobbuffet, feebas, slakoth, zigzagoon]).
```

These two facts are here to create trainers, their respective teams and associate them with the predicate `trainerwithteam`. As you can see, the two trainers have grown their team considerably since our early examples, their battle will be legendary !  
Ideally, these two facts should better be on the `pokemon.pl`file, but we wrote them in the `pokemon_rules.pl`at the same time we wrote the other rules.  

```prolog
totstats(Id, Totstat) :-
    stat(Id, 'hp', Hp),
    stat(Id, 'attack', Atk),
    stat(Id, 'special-attack', SpAtk),
    stat(Id, 'special-defense', SpDef),
    stat(Id, 'defense', Def),
    stat(Id, 'speed', Spd),
    Totstat is Hp + Atk + SpAtk + Def + SpDef + Spd.
```

We will base our pokemon battles on the sum of their stats, to make things easier we create the `totstats/2` rule to make the process easier.

We create then the rule `battleteam/5`which makes two different teams battle each other. To do so, recursion and severeal different clauses are required, let's start with the base clauses:  
```prolog
battleteam(Team1, Team2, team1wins, _, _) :- Team2 = [],  Team1 \= [].
battleteam(Team1, Team2, team2wins, _, _) :- Team1 = [],  Team2 \= [].
battleteam(Team1, Team2, draw, _, _) :- Team1 = [],  Team2 = [].
```

Here, you can see that if a team is empty while the other still has pokemons, the team with remaining pokemons wins (unless both are empty, in that case it is a draw). Seeing this, you can catch the idea that we want pokemons to battle each other, and if a pokemon is beaten by another, it is removed from the team of active/non-ko pokemon.  

Let's now tackle the recursive clauses. We are going to make the first pokemons of the team battle each other. The winning Pokemon is the pokemon with the higher stats. The losing pokemon is ko and the winning pokemon will battle the following pokemon of the team. The losing team is the first one to have all its pokemon ko (in that case, the list is empty). We also add another rule that a pokemon can't participate in more than 3 fights in a row, after that it will faint as well.
```prolog
battleteam([HT1|TT1], [HT2|TT2], Winningteam, C1, _) :-
    pokemon(Id1, HT1),
    totstats(Id1, Stat1),
    pokemon(Id2, HT2),
    totstats(Id2, Stat2),
    Stat1 > Stat2,
    C1 < 2,
    C1Next is C1 + 1,
    battleteam([HT1|TT1], TT2, Winningteam, C1Next, 0).

battleteam([HT1|TT1], [HT2|TT2], Winningteam, C1, _) :-
    pokemon(Id1, HT1),
    totstats(Id1, Stat1),
    pokemon(Id2, HT2),
    totstats(Id2, Stat2),
    Stat1 > Stat2,
    C1 >= 2,
    battleteam(TT1, TT2, Winningteam, 0, 0).

battleteam([HT1|TT1], [HT2|TT2], Winningteam, _, C2) :-
    pokemon(Id1, HT1),
    totstats(Id1, Stat1),
    pokemon(Id2, HT2),
    totstats(Id2, Stat2),
    Stat1 < Stat2,
    C2 < 2,
    C2Next is C2 + 1,
    battleteam(TT1, [HT2|TT2], Winningteam, 0, C2Next).

battleteam([HT1|TT1], [HT2|TT2], Winningteam, _, C2) :-
    pokemon(Id1, HT1),
    totstats(Id1, Stat1),
    pokemon(Id2, HT2),
    totstats(Id2, Stat2),
    Stat1 < Stat2,
    C2 >= 2,
    battleteam(TT1, TT2, Winningteam, 0, 0).

battleteam([HT1|TT1], [HT2|TT2], Winningteam, _, _) :-
    pokemon(Id1, HT1),
    totstats(Id1, Stat1),
    pokemon(Id2, HT2),
    totstats(Id2, Stat2),
    Stat1 =:= Stat2,
    battleteam(TT1, TT2, Winningteam, 0, 0).
```


We have 5 different clauses, corresponding each to a different case:
- Pokemon from team 1 beats Pokemon from team 2, it has not yet participated in 3 fights in a row, Pokemon from team2 faints
- Pokemon from team 1 beats Pokemon from team 2, this fight was his third fight, both Pokemon faint
- Pokemon from team 2 beats Pokemon from team 1, it has not yet participated in 3 fights in a row, Pokemon from team1 faints
- Pokemon from team 2 beats Pokemon from team 1, this fight was his third fight, both Pokemon faint
- Pokemon are of equal strength, they both faint 

To compute the number of fights in a row a pokemon participates in, we create two variables to keep track of it. When their values are not used in the clause, we use the anonymous variable `_` to indicate that information. This variable is reset to 0 if the pokemon faints, else it is increased by one after each fight.  


Lastly, we are creating a rule `trainersbattle/3` indicating the winner of a battle between two trainers. In the clause, we are retrieving the teams of the trainers and making them battle each other.  
Three cases are considered:
- Trainer 1 wins the battle
- Trainer 2 wins the battle
- The battle ends up in a draw

```prolog
trainersbattle(Trainer1, Trainer2, Trainer1) :-
    trainerwithteam(Trainer1, Team1),
    trainerwithteam(Trainer2, Team2),
    battleteam(Team1, Team2, team1wins, 0, 0), !.

trainersbattle(Trainer1, Trainer2, Trainer2) :-
    trainerwithteam(Trainer1, Team1),
    trainerwithteam(Trainer2, Team2),
    battleteam(Team1, Team2, team2wins, 0, 0), !.

trainersbattle(Trainer1, Trainer2, draw) :-
    trainerwithteam(Trainer1, Team1),
    trainerwithteam(Trainer2, Team2),
    battleteam(Team1, Team2, draw, 0, 0), !.
```

You can now query the knowledge base to see who would win between Thibaut and Alexandre:

In [ ]:
from pyswip import Prolog
prolog = Prolog()

prolog.consult("pokemon.pl")
prolog.consult("pokemon_rules.pl")

In [ ]:
battle = prolog.query("trainersbattle(thib, alex, Winner)")
for winner in battle:
    print(winner)

# References for the tutorial
- https://www.swi-prolog.org/pldoc/man?predicate=time%2F1
- https://www.swi-prolog.org/pldoc/man?predicate=statistics%2F2
- https://www.swi-prolog.org/man/clpfd.html
- https://www.swi-prolog.org/pldoc/man?section=defmodule
- https://swi-prolog.discourse.group/t/whats-the-difference-between-prolog-and-datalog/3604?utm_source=chatgpt.com


## Advantages and disadvantages of Prolog

### Advantages

* **Declarative programming:**
  Allows logic to be expressed in the form of rules and facts, simplifying reading and maintenance.
* **Automatic search for solutions:**
  Unification and backtracking automatically explore different solutions to queries.
* **Constraint-based programming:**
  Integration of CLP(FD) to easily solve problems with numerical constraints.

### Disadvantages

* **Limited performance for large volumes:**
  Less optimised than RDBMS for massive data processing.
* **Learning curve:**
  Declarative paradigm and backtracking logic can be confusing for beginners.
* **Execution control:**
  Requires judicious use of the ‘cut’ operator to avoid inefficient searches.


## Real-life uses of Prolog

* **[Medical diagnostics](https://en.wikipedia.org/wiki/Logic_Programming_Associates):** Logic Programming Associates has developed Allergenius, an expert system for interpreting allergen microarray results, written in Prolog. 
* **[Natural language processing](https://en.wikipedia.org/wiki/Prolog#Use_in_industry):** IBM Watson uses Prolog for pattern matching on syntax trees during question answering.

* **[Knowledge graph management](https://en.wikipedia.org/wiki/TerminusDB):** TerminusDB, an open-source knowledge base, is implemented in Prolog and Rust.


# Conclusion

This tutorial has shown how to install and configure Prolog on different operating systems, detailing the tools and commands needed to get started quickly. We explored the fundamental concepts of Prolog unification, backtracking, logical operators, lists and constraint programming, illustrating them with a concrete example based on the Pokémon database. This example enabled us to move progressively from simple queries to logical combinations, recursive searches and the use of numerical constraints, demonstrating the power and flexibility of the declarative paradigm. We also highlighted both the advantages of code readability, the automatic search engine and the CLP(FD) library, and the limitations of high-volume performance and the need to master Prolog's ‘cut’ operator. Finally, we have presented real-world uses in medical diagnostics, natural language processing and knowledge graph management, highlighting the impact of Prolog in industry and research. We hope that this guide has given you a clear understanding of Prolog and encouraged you to explore its extensions and integrations in your future projects.


## Project process

This project was conducted collaboratively. All phases of the work, from initial research to final writing, were carried out jointly by both of us, ensuring coherence and a consistent shared understanding of the topic.

## AI use 
For the preparation of this report, we used DeepL Translator for translating certain sections into English. \
Additionally, we utilized ChatGPT's deep search feature to assist in identifying sources and to improve the formulation .

# References

- https://www.swi-prolog.org/pldoc/man?section=ssu   
- https://pyswip.readthedocs.io/en/stable/get_started.html  
- https://pypi.org/project/pyswip/0.2.2/#:~:text=  
- https://pypi.org/project/swiplserver#:~:text=from%20swiplserver%20import%20PrologMQI%2C%20PrologThread  
- https://sut-ai.github.io/supplementary/notebooks/logic_programming#:~:text=There%20are%20some%20jupyter%20notebook,simple%20pip%20install%20should%20suffice  
- https://lpn.swi-prolog.org/lpnpage.php?pagetype=html&pageid=lpn-htmlse1#:~:text=There%20are%20only%20three%20basic,relationships%20that%20we%20find%20interesting  
- https://athena.ecs.csus.edu/~mei/logicp/conjunction-backtracking.html#:~:text=Prolog%20repeatedly%20tries%20to%20find,will%20be%20answered%20as%20failure
- https://www.cs.uleth.ca/~gaur/post/prolog-lecture/#:~:text=Given%20the%20database%20of%20related,han_solo
- https://swish.swi-prolog.org/example/movies.pl#:~:text=Part%201%3A%20Write%20queries%20to,the%20movies%20are
- https://lpn.swi-prolog.org/lpnpage.php?pageid=online
- https://www.rangakrish.com/index.php/2022/10/06/why-learn-prolog/
- https://www.swi-prolog.org/pldoc/man?section=clp